# اختبار MADCrowner على Google Colab GPU (عينة واحدة)

> **تنبيه خصوصية مهم:** تشغيل هذا الدفتر يرفع ملفات الحالة من Google Drive إلى جلسة Google Colab السحابية لمعالجتها. استخدمه فقط إذا كان مسموحًا لك بنقل هذه العينة إلى Google. لا يحتوي المستودع على ملفات `crown.ply` و`pna_crop.ply`؛ لا ترفع meshes سريرية إلى GitHub.

هذا فحص تقني فقط، وليس تدريبًا صالحًا للاستخدام السريري. لا توجد أوزان MADCrowner مدرّبة في المستودع؛ يبدأ الاختبار بأوزان عشوائية ويحفظ checkpoint تجريبيًا لا يصلح لتصميم تاج. سيشغّل الدفتر `IOS_Dataset` مع PyTorch3D FPS الأصلي، ثم forward/backward وتحديث optimizer عشر مرات على العينة نفسها.

## قبل التشغيل
1. من Colab اختر **Runtime → Change runtime type → T4 GPU** (إذا كان متاحًا لحسابك).
2. أضف GitHub token بصلاحية **Contents: Read-only** إلى Colab Secrets باسم `GITHUB_TOKEN`، أو أدخله عند مطالبة الدفتر. لا تضع الرمز داخل خلية أو ترفعه إلى GitHub.
3. أنشئ في Google Drive مجلد `MyDrive/DentalAI/`، وضع داخله `case_46_candidate_01.zip` الذي يحتوي الملفات الخمسة: `crown.ply`, `pna_crop.ply`, `boundary_labels.npy`, `crown_attributes.h5`, و`psr_my.npz`. يمكن أن تكون الملفات في جذر ZIP أو داخل مجلد واحد.
4. شغّل الخلايا بالترتيب. تثبيت PyTorch3D من المصدر قد يستغرق عدة دقائق. إذا لم يتوفر T4 أو لم تسمح قيود Colab ببناء الامتداد، سيتوقف الدفتر برسالة خطأ واضحة بدل الادعاء بنجاح FPS.

المراجع: [تعليمات تثبيت PyTorch3D](https://github.com/facebookresearch/pytorch3d/blob/main/INSTALL.md) و[دليل PyTorch لاستخدام Google Colab وT4](https://docs.pytorch.org/tutorials/beginner/colab.html).

In [ ]:
import os, sys, subprocess
print('Python:', sys.version.split()[0])
gpu_info = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True, check=True).stdout.strip()
print('GPU:', gpu_info)
if 'T4' not in gpu_info.upper():
    raise RuntimeError('لم يتم تخصيص NVIDIA T4. اختر Runtime > Change runtime type > T4 GPU ثم أعد التشغيل.')

## 1) استنساخ المستودع الخاص
يستخدم هذا القسم header مؤقتًا في عملية `git clone` فقط؛ لا يكتب الرمز في عنوان remote أو ملفات المستودع. إذا استُخدم Colab Secret، فعّل خيار Notebook access لذلك السر.

In [ ]:
import base64, getpass, subprocess
from pathlib import Path
try:
    from google.colab import userdata
    gh_token = userdata.get('GITHUB_TOKEN')
except Exception:
    gh_token = None
if not gh_token:
    gh_token = getpass.getpass('GitHub token (Contents: Read-only): ')
if not gh_token:
    raise RuntimeError('يلزم GitHub token لقراءة المستودع الخاص.')
REPO_DIR = Path('/content/dental-crown-ai')
if not REPO_DIR.exists():
    basic = base64.b64encode(('x-access-token:' + gh_token).encode()).decode()
    env = os.environ.copy()
    env['GIT_CONFIG_COUNT'] = '1'
    env['GIT_CONFIG_KEY_0'] = 'http.extraheader'
    env['GIT_CONFIG_VALUE_0'] = 'AUTHORIZATION: basic ' + basic
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/eliasajeep1966-gif/dental-crown-ai.git', str(REPO_DIR)], env=env, check=True)
    del env, basic
gh_token = None
print('Repository ready:', REPO_DIR)

## 2) تثبيت إصدار معروف من PyTorch ثم بناء PyTorch3D لـCUDA
نثبت PyTorch 2.4.1 + CUDA 12.1 لتقليل اختلافات API مع كود المشروع. يُبنى PyTorch3D من مصدره الرسمي، وتُحمّل wheels المطابقة لـ`torch-scatter` و`torch-cluster`. هذه تبعيات مسار smoke test وليست تثبيتًا كاملًا لتدريب MADCrowner طويلًا. إذا تغيّرت صورة Colab أو تعذر البناء، فهذا فشل توافق بيئة وليس نتيجة اختبار للمشروع.

In [ ]:
import subprocess, sys, os
def pip_install(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)
# لا نستورد torch قبل هذه الخطوة حتى يستخدم بناء PyTorch3D الحزمة المثبتة.
pip_install('--index-url', 'https://download.pytorch.org/whl/cu121', 'torch==2.4.1', 'torchvision==0.19.1', 'torchaudio==2.4.1')
pip_install('numpy==1.26.4', 'scipy==1.15.3', 'h5py==3.14.0', 'trimesh==4.8.1', 'PyYAML==6.0.2', 'tqdm==4.67.1', 'imageio==2.37.0', 'plyfile==1.1.2', 'scikit-image==0.25.2', 'open3d==0.19.0', 'libigl', 'ipdb', 'fvcore', 'iopath', 'torch-geometric==2.6.1')
pip_install('torch-scatter', 'torch-cluster', '-f', 'https://data.pyg.org/whl/torch-2.4.1+cu121.html')
env = os.environ.copy()
env['FORCE_CUDA'] = '1'
env['MAX_JOBS'] = '2'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation', 'git+https://github.com/facebookresearch/pytorch3d.git@stable'], env=env, check=True)
print('Smoke-test dependencies installed. If this cell reports a build error, keep the complete error for diagnosis.')

In [ ]:
import torch, pytorch3d
from pytorch3d.ops import sample_farthest_points
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('PyTorch3D import:', pytorch3d.__file__)
assert torch.cuda.is_available()
probe = torch.rand(1, 32, 3, device='cuda')
sampled, indices = sample_farthest_points(probe, K=8)
assert sampled.shape == (1, 8, 3) and indices.shape == (1, 8)
print('Native CUDA FPS probe passed:', tuple(sampled.shape))

## 3) تحميل ملفات الحالة من Google Drive
لا يقرأ هذا القسم ملفات العينة من GitHub. ضع ZIP في Google Drive كما هو موضح أعلى الدفتر؛ إذا لم يجده، سيطلب منك رفعه من اللابتوب. تبقى نسخة العمل داخل جلسة Colab المؤقتة.

In [ ]:
from google.colab import drive
import zipfile, shutil
drive.mount('/content/drive')
SAMPLE_ZIP = Path('/content/drive/MyDrive/DentalAI/case_46_candidate_01.zip')
if not SAMPLE_ZIP.is_file():
    from google.colab import files
    print('لم أجد ZIP في Drive؛ اختر case_46_candidate_01.zip من اللابتوب لرفعه إلى Colab.')
    uploaded = files.upload()
    candidates = [name for name in uploaded if name.endswith('.zip')]
    if len(candidates) != 1:
        raise FileNotFoundError('ارفع ZIP واحدًا بالضبط يحتوي ملفات العينة الخمسة.')
    SAMPLE_ZIP = Path('/content') / candidates[0]
required = ['crown.ply', 'pna_crop.ply', 'boundary_labels.npy', 'crown_attributes.h5', 'psr_my.npz']
extract_dir = Path('/content/case46_upload')
if extract_dir.exists():
    shutil.rmtree(extract_dir)
extract_dir.mkdir(parents=True)
with zipfile.ZipFile(SAMPLE_ZIP) as zf:
    members = zf.namelist()
    if any(Path(m).is_absolute() or '..' in Path(m).parts for m in members):
        raise ValueError('ZIP يحتوي مسارًا غير آمن؛ أوقفنا فك الضغط.')
    zf.extractall(extract_dir)
found = {}
for name in required:
    matches = list(extract_dir.rglob(name))
    if len(matches) != 1:
        raise FileNotFoundError(f'توقعنا ملفًا واحدًا باسم {name} داخل ZIP، وجدنا {len(matches)}')
    found[name] = matches[0]
DATA_ROOT = Path('/content/madcrowner_onecase')
CASE_DIR = DATA_ROOT / '46' / 'train' / 'case_46_candidate_01'
CASE_DIR.mkdir(parents=True, exist_ok=True)
for name, src in found.items():
    shutil.copy2(src, CASE_DIR / name)
print('Loaded exactly five sample files into:', CASE_DIR)
print('Files:', ', '.join(sorted(p.name for p in CASE_DIR.iterdir())))

## 4) اختبار IOS_Dataset مع PyTorch3D FPS الأصلي
نستخدم عدد نقاط أقل في هذا smoke test لتقليل احتمال نفاد ذاكرة T4. هذا يختبر قراءة البيانات وامتداد FPS الأصلي، ولا يغيّر إعداد التدريب النهائي.

In [ ]:
import sys, numpy as np, torch
MAD_ROOT = REPO_DIR / 'third_party' / 'MADCrowner'
sys.path.insert(0, str(MAD_ROOT))
from mydataset.DentalDataset import IOS_Dataset
dataset = IOS_Dataset(DATA_ROOT, is_train=True, sample_points=64)
assert len(dataset) == 1, f'Expected one case, got {len(dataset)}'
batch = next(iter(torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0)))
inputs, targets, crown_center, template, psr_grid, curvatures, margin, case_path = batch
assert inputs.shape == (1, 128, 4)
assert targets.shape == (1, 256, 3)
assert template.shape == (1, 64, 3)
assert psr_grid.shape == (1, 128, 128, 128)
assert torch.isfinite(inputs).all() and torch.isfinite(targets).all()
assert torch.isfinite(curvatures).all() and torch.isfinite(margin).all()
print('IOS_Dataset/FPS OK:', {'inputs': tuple(inputs.shape), 'crown': tuple(targets.shape), 'template': tuple(template.shape), 'psr': tuple(psr_grid.shape), 'positive_sampled_labels': int(inputs[..., 3].sum().item())})

## 5) عشر خطوات تدريب تجريبية على CrownDeformer
يستخدم الاختبار resolution أصغر للـPSR (`32³`) وعدد نقاط محدود حتى يناسب T4. يُعاد تحجيم target PSR إلى هذا المقاس. هذه ليست إعدادات تدريب نهائي ولا تقيس جودة التاج.

In [ ]:
import torch.nn.functional as F
from pytorch3d.loss import chamfer_distance
from pytorch3d.ops import sample_farthest_points
from models.crowndeformer import CrownDeformer
from models.loss import curvature_penalty_loss
device = torch.device('cuda')
model = CrownDeformer(psr_resolution=32).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-5)
inputs = inputs.to(device)
targets = targets.to(device)
template = template.to(device)
psr_grid = psr_grid.to(device)
curvatures = curvatures.to(device)
margin = margin.to(device)
target_psr = F.interpolate(psr_grid.unsqueeze(1), size=(32, 32, 32), mode='trilinear', align_corners=False).squeeze(1)
target_seed = sample_farthest_points(targets.contiguous(), K=64)[0]
first_param = next(p for p in model.parameters() if p.requires_grad)
before = first_param.detach().clone()
loss_history = []
model.train()
for step in range(1, 11):
    optimizer.zero_grad(set_to_none=True)
    pred_psr, seed, coarse_pc, dense_cloud = model(inputs, template)
    loss_psr = F.mse_loss(pred_psr, target_psr)
    loss_margin_curvature = curvature_penalty_loss(dense_cloud, targets, curvatures, margin, weight_lambda=1.0)
    loss_seed = chamfer_distance(seed, target_seed)[0]
    loss_coarse = chamfer_distance(coarse_pc, targets)[0]
    loss = loss_psr + loss_margin_curvature + loss_seed + loss_coarse
    if not torch.isfinite(loss):
        raise FloatingPointError(f'Non-finite loss at step {step}')
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    loss_history.append(float(loss.detach().cpu()))
    print(f'step {step:02d}/10 | loss={loss_history[-1]:.6f} | peak GPU memory={torch.cuda.max_memory_allocated()/1024**3:.2f} GiB')
changed = not torch.equal(before, first_param.detach())
assert changed, 'Optimizer did not update the checked trainable parameter.'
print('Ten optimizer steps completed; parameter update confirmed.')

## 6) حفظ checkpoint تجريبي على Google Drive
الملف الناتج يحفظ حالة النموذج والـoptimizer بعد 10 خطوات. احتفظ به بوصفه **checkpoint لاختبار أن المسار يعمل فقط**؛ لا تستخدمه للاستدلال السريري.

In [ ]:
checkpoint_dir = Path('/content/drive/MyDrive/DentalAI/checkpoints')
checkpoint_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = checkpoint_dir / 'case46_colab_smoke_10steps.pt'
torch.save({'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(), 'steps': 10, 'loss_history': loss_history, 'note': 'smoke test only; random initialization; not for clinical use'}, checkpoint_path)
check = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
assert check['steps'] == 10 and len(check['loss_history']) == 10
print('Checkpoint saved and reloaded:', checkpoint_path)
print('Final loss:', check['loss_history'][-1])
print('Checkpoint size (MiB):', round(checkpoint_path.stat().st_size / 1024**2, 2))